# Exploratory Data Analysis — BAF Base Dataset

**Dataset:** Bank Account Fraud (BAF) Base — NeurIPS 2022 (Jesus et al.)

**Purpose:** Validate data quality, understand feature distributions, assess feature relevance for fraud detection, and decide which columns to keep/remove before model training.

**Sections:**
1. Setup & Loading
2. Missing Values
3. Class Distribution
4. Descriptive Statistics
5. Feature Distributions
6. Correlation Analysis
7. Feature–Target Association
8. Mutual Information
9. Outlier Analysis
10. Column `month` Analysis
11. Conclusions & Decisions

## 1. Setup & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = Path("..") / "datasets"
df = pd.read_csv(DATA_DIR / "Base.csv")

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names ({len(df.columns)}):\n{list(df.columns)}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
df.head()

In [ ]:
# Separate feature types
TARGET = "fraud_bool"
CATEGORICAL = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
TEMPORAL = ["month"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + [TARGET] + TEMPORAL]

print(f"Target: {TARGET}")
print(f"Categorical ({len(CATEGORICAL)}): {CATEGORICAL}")
print(f"Temporal ({len(TEMPORAL)}): {TEMPORAL}")
print(f"Numeric ({len(NUMERIC)}): {NUMERIC}")

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing Count", ascending=False)

if missing_df.empty:
    print("✓ No missing values in any column.")
else:
    print(f"⚠ {len(missing_df)} columns with missing values:")
    display(missing_df)

print(f"\nTotal cells: {df.shape[0] * df.shape[1]:,}")
print(f"Total missing: {df.isnull().sum().sum():,}")

## 3. Class Distribution

In [ ]:
class_counts = df[TARGET].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(3)

print("Class distribution:")
for label, count in class_counts.items():
    print(f"  {label} ({'Fraud' if label == 1 else 'Legit'}): {count:>10,}  ({class_pct[label]:.3f}%)")
print(f"\nImbalance ratio (legit:fraud): {class_counts[0] / class_counts[1]:.1f}:1")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = axes[0].bar(["Legit (0)", "Fraud (1)"], class_counts.values,
                   color=["steelblue", "crimson"])
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{count:,}", ha="center", va="bottom", fontsize=10)
axes[0].set_title("Class Counts")
axes[0].set_ylabel("Count")

# Pie chart
axes[1].pie(class_counts.values, labels=["Legit (0)", "Fraud (1)"],
            autopct="%1.2f%%", colors=["steelblue", "crimson"], startangle=90)
axes[1].set_title("Class Proportion")

plt.suptitle("BAF Base — Class Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Descriptive Statistics

In [ ]:
print("=" * 60)
print("NUMERIC FEATURES — Descriptive Statistics")
print("=" * 60)
df[NUMERIC].describe().T.round(4)

In [ ]:
print("=" * 60)
print("CATEGORICAL FEATURES — Value Counts")
print("=" * 60)
for col in CATEGORICAL:
    print(f"\n--- {col} ({df[col].nunique()} unique) ---")
    vc = df[col].value_counts()
    vc_pct = (vc / len(df) * 100).round(2)
    display(pd.DataFrame({"Count": vc, "%": vc_pct}))

## 5. Feature Distributions

In [ ]:
# Numeric feature distributions — split by class
n_cols = 4
n_rows = int(np.ceil(len(NUMERIC) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(NUMERIC):
    ax = axes[i]
    for label, color, name in [(0, "steelblue", "Legit"), (1, "crimson", "Fraud")]:
        subset = df.loc[df[TARGET] == label, col]
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

# Hide unused axes
for j in range(len(NUMERIC), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("BAF Base — Numeric Feature Distributions (by class)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature distributions — fraud rate per category
fig, axes = plt.subplots(1, len(CATEGORICAL), figsize=(5 * len(CATEGORICAL), 5))
if len(CATEGORICAL) == 1:
    axes = [axes]

for i, col in enumerate(CATEGORICAL):
    ax = axes[i]
    fraud_rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    bars = ax.bar(range(len(fraud_rate)), fraud_rate.values, color="crimson", alpha=0.7)
    ax.set_xticks(range(len(fraud_rate)))
    ax.set_xticklabels(fraud_rate.index, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Fraud Rate")
    ax.set_title(col, fontsize=11)
    ax.axhline(y=df[TARGET].mean(), color="gray", linestyle="--", linewidth=1, label="Overall rate")
    ax.legend(fontsize=8)
    # Annotate bars
    for bar, val in zip(bars, fraud_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("BAF Base — Fraud Rate by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Pearson correlation matrix for numeric features
corr = df[NUMERIC].corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, annot_kws={"size": 7})
plt.title("BAF Base — Pearson Correlation Matrix (Numeric Features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated pairs (|r| > 0.7)
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr_pairs.append({
                "Feature 1": corr.columns[i],
                "Feature 2": corr.columns[j],
                "Pearson r": round(r, 4)
            })

if high_corr_pairs:
    hc_df = pd.DataFrame(high_corr_pairs).sort_values("Pearson r", key=abs, ascending=False)
    print(f"⚠ {len(hc_df)} feature pairs with |r| > 0.70:")
    display(hc_df.reset_index(drop=True))
else:
    print("✓ No numeric feature pairs with |r| > 0.70.")

## 7. Feature–Target Association

In [ ]:
# Point-biserial correlation: each numeric feature vs binary target
from scipy.stats import pointbiserialr

pb_results = []
for col in NUMERIC:
    r, p = pointbiserialr(df[TARGET], df[col])
    pb_results.append({"Feature": col, "Point-Biserial r": round(r, 4), "p-value": p})

pb_df = pd.DataFrame(pb_results).sort_values("Point-Biserial r", key=abs, ascending=False)
pb_df["Significant (p<0.001)"] = pb_df["p-value"] < 0.001
display(pb_df.reset_index(drop=True))

# Horizontal bar chart
plt.figure(figsize=(10, 8))
colors = ["crimson" if r > 0 else "steelblue" for r in pb_df["Point-Biserial r"]]
plt.barh(pb_df["Feature"], pb_df["Point-Biserial r"].abs(), color=colors, alpha=0.7)
plt.xlabel("|Point-Biserial r|")
plt.title("BAF Base — Feature–Target Correlation (|Point-Biserial r|)", fontsize=13, fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: top 8 most correlated numeric features vs target
top8 = pb_df.head(8)["Feature"].tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(top8):
    ax = axes[i]
    df.boxplot(column=col, by=TARGET, ax=ax,
              boxprops=dict(linewidth=1.5),
              medianprops=dict(color="crimson", linewidth=2))
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("fraud_bool")
    ax.set_ylabel("")

plt.suptitle("BAF Base — Top 8 Features vs Fraud (Boxplots)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# Encode categoricals for MI computation
df_mi = df.drop(columns=TEMPORAL).copy()
le_dict = {}
for col in CATEGORICAL:
    le = LabelEncoder()
    df_mi[col] = le.fit_transform(df_mi[col])
    le_dict[col] = le

X_mi = df_mi.drop(columns=[TARGET])
y_mi = df_mi[TARGET]

# Compute MI (discrete_features mask for categoricals)
discrete_mask = [col in CATEGORICAL for col in X_mi.columns]
mi_scores = mutual_info_classif(X_mi, y_mi, discrete_features=discrete_mask, random_state=42, n_neighbors=5)

mi_df = pd.DataFrame({"Feature": X_mi.columns, "MI Score": mi_scores})
mi_df = mi_df.sort_values("MI Score", ascending=False).reset_index(drop=True)
mi_df["Type"] = mi_df["Feature"].apply(lambda x: "Categorical" if x in CATEGORICAL else "Numeric")

display(mi_df)

# Bar chart
plt.figure(figsize=(10, 8))
colors = ["darkorange" if t == "Categorical" else "steelblue" for t in mi_df["Type"]]
plt.barh(mi_df["Feature"], mi_df["MI Score"], color=colors, alpha=0.8)
plt.xlabel("Mutual Information Score")
plt.title("BAF Base — Mutual Information with Target (fraud_bool)", fontsize=13, fontweight="bold")
plt.gca().invert_yaxis()

# Legend
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color="steelblue", alpha=0.8, label="Numeric"),
                     Patch(color="darkorange", alpha=0.8, label="Categorical")],
           loc="lower right")
plt.tight_layout()
plt.show()

# Low MI features
low_mi = mi_df[mi_df["MI Score"] < 0.001]
if not low_mi.empty:
    print(f"\n⚠ Features with MI < 0.001 (near-zero relevance):")
    display(low_mi)
else:
    print("\n✓ All features have MI ≥ 0.001.")

## 9. Outlier Analysis

In [ ]:
# IQR-based outlier detection for numeric features
outlier_stats = []
for col in NUMERIC:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_stats.append({
        "Feature": col,
        "Q1": round(Q1, 4),
        "Q3": round(Q3, 4),
        "IQR": round(IQR, 4),
        "Lower Bound": round(lower, 4),
        "Upper Bound": round(upper, 4),
        "Outliers": n_outliers,
        "Outlier %": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_stats).sort_values("Outlier %", ascending=False)
display(outlier_df.reset_index(drop=True))

# Bar chart of outlier percentages
plt.figure(figsize=(10, 8))
plt.barh(outlier_df["Feature"], outlier_df["Outlier %"], color="coral", alpha=0.7)
plt.xlabel("Outlier %")
plt.title("BAF Base — Outlier Percentage per Feature (IQR method)", fontsize=13, fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Are outliers more common in fraud or legit?
print("Outlier proportion by class (top features with most outliers):")
print("=" * 65)
top_outlier_features = outlier_df.head(10)["Feature"].tolist()

class_outlier_data = []
for col in top_outlier_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    is_outlier = (df[col] < lower) | (df[col] > upper)

    for label, name in [(0, "Legit"), (1, "Fraud")]:
        mask = df[TARGET] == label
        n_class = mask.sum()
        n_outlier_class = (is_outlier & mask).sum()
        class_outlier_data.append({
            "Feature": col,
            "Class": name,
            "Outliers in Class": n_outlier_class,
            "Class Size": n_class,
            "Outlier Rate in Class %": round(n_outlier_class / n_class * 100, 2)
        })

co_df = pd.DataFrame(class_outlier_data)
display(co_df.pivot_table(index="Feature", columns="Class",
                          values="Outlier Rate in Class %").round(2))

## 10. Column `month` Analysis

In [ ]:
# Analyse the 'month' column — should we keep or drop it?
print(f"Unique values: {sorted(df['month'].unique())}")
print(f"Value counts:")
display(df['month'].value_counts().sort_index())

# Fraud rate by month
fraud_by_month = df.groupby('month')[TARGET].agg(['sum', 'count', 'mean'])
fraud_by_month.columns = ['Frauds', 'Total', 'Fraud Rate']
display(fraud_by_month)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Counts by month
month_counts = df['month'].value_counts().sort_index()
axes[0].bar(month_counts.index, month_counts.values, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Count")
axes[0].set_title("Samples per Month")

# Fraud rate by month
axes[1].plot(fraud_by_month.index, fraud_by_month['Fraud Rate'], 'o-', color='crimson')
axes[1].axhline(y=df[TARGET].mean(), color='gray', linestyle='--', label='Overall rate')
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Fraud Rate")
axes[1].set_title("Fraud Rate by Month")
axes[1].legend()

plt.suptitle("BAF Base — Month Column Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n→ 'month' is a temporal artefact from the CTGAN data generation process.")
print("  It should NOT be used as a feature — it leaks temporal information")
print("  about data synthesis, not about real applicant behaviour.")
print("  DECISION: Drop 'month' (already implemented in the pipeline).")

## 11. Conclusions & Decisions

In [ ]:
print("" + "=" * 70)
print(" EDA SUMMARY — BAF Base Dataset")
print("=" * 70)

print(f"\n1. DATASET SIZE: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical, 1 temporal, 1 target)")

print(f"\n2. MISSING VALUES: {df.isnull().sum().sum()} — dataset is complete.")

fraud_rate = df[TARGET].mean()
print(f"\n3. CLASS IMBALANCE: {fraud_rate:.4f} ({fraud_rate*100:.2f}% fraud)")
print(f"   Ratio: {(1-fraud_rate)/fraud_rate:.1f}:1 (legit:fraud)")

if high_corr_pairs:
    print(f"\n4. HIGH CORRELATION: {len(high_corr_pairs)} feature pair(s) with |r| > 0.70")
    for pair in high_corr_pairs:
        print(f"   • {pair['Feature 1']} ↔ {pair['Feature 2']}: r = {pair['Pearson r']}")
else:
    print(f"\n4. HIGH CORRELATION: None with |r| > 0.70")

print(f"\n5. TOP 5 FEATURES by Mutual Information:")
for _, row in mi_df.head(5).iterrows():
    print(f"   • {row['Feature']:30s}  MI = {row['MI Score']:.4f}  ({row['Type']})")

print(f"\n6. LOWEST MI FEATURES (candidates for removal if MI ≈ 0):")
for _, row in mi_df.tail(5).iterrows():
    print(f"   • {row['Feature']:30s}  MI = {row['MI Score']:.4f}  ({row['Type']})")

print(f"\n7. COLUMN 'month': Dropped (temporal artefact, not a real feature).")

print("\n" + "=" * 70)
print(" DECISIONS")
print("=" * 70)
print("""
• Drop 'month' — confirmed as CTGAN temporal artefact.
• Keep all other features for now — review after seeing correlation
  and MI results above.
• No imputation needed — zero missing values.
• Class imbalance confirmed (~1.1% fraud) — justifies the use of
  resampling strategies and threshold optimization in the pipeline.
""")